# AHFL Current Version — PaddleOCR 3.4.0 + PaddlePaddle 3.3.0 GPU (cu121)
Tests `/content/drive/MyDrive/AHFL-GPU-Tests/ahfl-working-Gpu` with images from `/Users/tusharjain/projects/AHFL/AHFL-GPU/ahfl-working-Gpu/imges`

In [ ]:
import sys, os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
PROJECT = '/content/drive/MyDrive/AHFL-GPU-Tests/ahfl-working-Gpu'
if not os.path.exists(PROJECT): raise FileNotFoundError(f'{PROJECT}')
sys.path.insert(0, PROJECT)
print(f'✓ Project: {PROJECT}')

In [ ]:
import subprocess
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'paddlepaddle-gpu==3.3.0',
        '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu121/'])
except:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'paddlepaddle==3.3.0'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'paddleocr==3.4.0', 'opencv-python-headless'])
import paddle, paddleocr
print(f'✓ PaddlePaddle {paddle.__version__} + PaddleOCR {paddleocr.__version__}')

In [ ]:
from paddleocr import PaddleOCR
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Current version: device="gpu:0" replaces removed use_gpu param
ocr = PaddleOCR(lang='en', use_textline_orientation=True, device='gpu:0', show_log=False)
print('✓ OCR ready (current: PaddleOCR 3.4.0 + PaddlePaddle 3.3.0)')

In [ ]:
from glob import glob
img_dir = '/Users/tusharjain/projects/AHFL/AHFL-GPU/ahfl-working-Gpu/imges'
images = sorted(glob(f'{img_dir}/*.{{png,jpg,jpeg}}'))
print(f'Found {len(images)} images')
if not images:
    from google.colab import files
    print('Upload images:')
    uploaded = files.upload()
    images = list(uploaded.keys())

results = {}
for img in images[:5]:
    name = os.path.basename(img)
    print(f'\n→ {name}')
    try:
        result = ocr.ocr(img)
        results[name] = result
        if result and result[0]:
            print(f'  {len(result[0])} regions')
            for i, line in enumerate(result[0][:2]):
                print(f'    "{line[1][0]}" ({line[1][1]:.3f})')
    except Exception as e:
        print(f'  ✗ {e}')

In [ ]:
import json
path = '/content/drive/MyDrive/AHFL-GPU-Tests/ahfl-working-Gpu'
out = {n: [{"text": line[1][0], "conf": round(line[1][1], 4)} for line in (r[0] or [])] for n,r in results.items()}
json_f = os.path.join(path, 'current_version_results.json')
with open(json_f, 'w') as f: json.dump(out, f, indent=2)
print(f'✓ Saved: current_version_results.json')
for f in sorted(os.listdir(path)):
    if os.path.isfile(os.path.join(path,f)): print(f'  • {f}')